# Word2vec example

In [8]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import torch
from collections import Counter
import os
import numpy as np
import os

import urllib.request

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(device)


mps


## Reading the training text

In [11]:


file_path = "../../Data/AI2.txt"
print("Reading the local file", file_path)
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

corpus = text_data.split(".")[:500] ## taking only a part of the text sentences

Reading the local file ../../Data/AI2.txt


In [13]:
corpus[:10]

['\n\nArtificial Intelligence: Understanding a Transformative Technology\n\nArtificial Intelligence, commonly referred to as AI, is one of the most influential technological developments of the 21st century',
 ' Often surrounded by fascination, fear, and misunderstanding, AI is neither magic nor science fiction',
 ' It is a set of scientific methods, mathematical models, and computer systems designed to enable machines to perform tasks that normally require human intelligence',
 ' These tasks include learning from experience, recognizing patterns, understanding language, making decisions, and solving complex problems',
 '\n\nWhat Is Artificial Intelligence?\n\nAt its core, Artificial Intelligence is the ability of a machine to simulate aspects of human intelligence',
 ' This does not mean that machines think or feel like humans, but rather that they can process information, adapt to new data, and act in ways that appear intelligent',
 ' AI systems are typically trained on large amounts

In [15]:
# ============================================================
# 2. Preprocessing
# ============================================================
def tokenize(corpus):
    return [sentence.lower().split() for sentence in corpus]

def build_vocab(tokens):
    counts = Counter(word for sent in tokens for word in sent)
    vocab = {word: i for i, word in enumerate(counts)}
    ivocab = {i: word for word, i in vocab.items()}
    return vocab, ivocab, counts

tokens = tokenize(corpus)
vocab, ivocab, word_counts = build_vocab(tokens)
vocab_size = len(vocab)

print("Vocab size=", vocab_size)
print(list(vocab.items())[:20])


Vocab size= 1296
[('artificial', 0), ('intelligence:', 1), ('understanding', 2), ('a', 3), ('transformative', 4), ('technology', 5), ('intelligence,', 6), ('commonly', 7), ('referred', 8), ('to', 9), ('as', 10), ('ai,', 11), ('is', 12), ('one', 13), ('of', 14), ('the', 15), ('most', 16), ('influential', 17), ('technological', 18), ('developments', 19)]


In [17]:


# ============================================================
# 3. Generate skip‑gram pairs
# ============================================================
def generate_skipgram_pairs(tokens, vocab, window_size=2):
    pairs = []
    for sentence in tokens:
        encoded = [vocab[w] for w in sentence]
        for i, center in enumerate(encoded):
            for j in range(max(0, i - window_size),
                           min(len(encoded), i + window_size + 1)):
                if i != j:
                    pairs.append((center, encoded[j]))
    return pairs






## Negative sampling distribution
* Use of power < 1
* Reduces dominance of very frequent words
* Still favors common words (important for language structure)
* Gives rare words a better chance to appear

In [20]:
# ============================================================
# 4. Negative sampling distribution
# ============================================================
def negative_sampling_dist(word_counts, vocab, power=0.75):
    freqs = torch.tensor([word_counts[word] for word in vocab])
    probs = freqs.float() ** power
    return probs / probs.sum()

neg_dist = negative_sampling_dist(word_counts, vocab)

# Word2Vec model

In [23]:

# ============================================================
# 5. Word2Vec Skip‑Gram model
# ============================================================
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)

    def forward(self, center, context, negatives):
        # center:   (B,)
        # context:  (B,)
        # negatives:(B, K)

        v_center = self.in_embed(center)               # (B, D)
        v_context = self.out_embed(context)             # (B, D)
        v_neg = self.out_embed(negatives)               # (B, K, D)

        pos_score = torch.sum(v_center * v_context, dim=1)
        pos_loss = torch.log(torch.sigmoid(pos_score))

        neg_score = torch.bmm(v_neg, v_center.unsqueeze(2)).squeeze(-1) ## batch matrix multiplication
        neg_loss = torch.log(torch.sigmoid(-neg_score)).sum(dim=1)

        return -(pos_loss + neg_loss).mean()

In [25]:

MODEL_PATH = "../../Data/model/W2V_model.pth"


if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
    model.eval()  # inference mode
    print("Model loaded. No learning required")
else:
    print("No saved model found. Learning from scratch.")
    pairs = generate_skipgram_pairs(tokens, vocab, window_size=3)

    # ============================================================
    # 6. Training
    # ============================================================
    EMBED_DIM = 50
    NEG_SAMPLES = 5
    EPOCHS = 200
    LR = 0.001

    model = Word2Vec(vocab_size, EMBED_DIM)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        total_loss = 0.0
        random.shuffle(pairs)

        for center, context in pairs:
          negatives = torch.multinomial(
              neg_dist, NEG_SAMPLES, replacement=True
          )

          center = torch.tensor([center])
          context = torch.tensor([context])
          negatives = negatives.unsqueeze(0)

          if random.random() < 0.00001:
            print("center=",list(vocab.items())[center.item()])
            print("context=",[list(vocab.items())[n]  for n in [context]])
            print("negatives=", [list(vocab.items())[n]  for n in negatives.tolist()[0]])

          loss = model(center, context, negatives)

          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

          total_loss += loss.item()


        print(f"Epoch {epoch:03d} | Loss: {total_loss:.4f}")

    # Save only the weights
    torch.save(model.state_dict(), MODEL_PATH)

    print(f"Model saved to {MODEL_PATH}")
    
embeddings = model.in_embed.weight.detach()
    


No saved model found. Learning from scratch.
center= ('concepts', 524)
context= [('engineering', 1133)]
negatives= [('rarely', 1065), ('investigate', 1257), ('governance', 999), ('meaningful', 561), ('are', 98)]
Epoch 000 | Loss: 312221.4879
center= ('that', 48)
context= [('impossible', 308)]
negatives= [('likely', 1155), ('to', 9), ('mechanisms,', 1010), ('loss', 543), ('referred', 8)]
Epoch 001 | Loss: 213370.0948
center= ('works', 154)
context= [('modern', 155)]
negatives= [('emerged', 686), ('issue', 337), ('effective', 653), ('explainability', 346), ('toolkit', 1023)]
center= ('understand', 823)
context= [('genuinely', 822)]
negatives= [('tool', 688), ('created', 508), ('constitutes', 1148), ('understanding', 2), ('from', 56)]
Epoch 002 | Loss: 154511.5559
Epoch 003 | Loss: 116146.5808
Epoch 004 | Loss: 89917.4649
center= ('motivation', 471)
context= [('the', 15)]
negatives= [('lack', 1079), ('envisioned', 481), ('awareness,', 1288), ('working', 412), ('the', 15)]
Epoch 005 | Loss

In [26]:
# ============================================================
# 7. Inspect embeddings
# ============================================================

def cosine_similarity(word1, word2):
    v1 = embeddings[vocab[word1]]
    v2 = embeddings[vocab[word2]]
    return F.cosine_similarity(v1, v2, dim=0).item()

print("\nCosine similarities:")
print("artificial vs intelligence:", cosine_similarity("artificial", "intelligence"))
print("computer vs economy:", cosine_similarity("computer", "economy"))
print("machine vs intelligence:", cosine_similarity("machine", "intelligence"))
print("computer vs machine:", cosine_similarity("computer", "machine"))




Cosine similarities:
artificial vs intelligence: 0.7212168574333191
computer vs economy: 0.1668638437986374
machine vs intelligence: 0.3489212989807129
computer vs machine: 0.27200278639793396


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE



ivocab = {i: word for word, i in vocab.items()}

#embeddings = model.in_embed.weight.detach()
words =[ 'artificial', 'intelligence:', 'computer', 'machine', 'learning',
        'behavior','economy','optimization','loss','error', 'structure','job','power']
X = embeddings.numpy()
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)
plt.figure(figsize=(8, 6))
#plt.figure(figsize=(.show()
plt.scatter(X_2d[:len(words), 0], X_2d[:len(words), 1], s=40)

for i, word in enumerate(words):
    plt.annotate(
        word,
        (X_2d[i, 0], X_2d[i, 1]),
        fontsize=10,
        alpha=0.8
    )

plt.title("Word2Vec embeddings (PCA projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.tight_layout()


In [ ]:
words

In [ ]:
vocab